# APD Vision — training detektor APD (`best.pt`)

Notebook ini melatih model deteksi 4 APD: **helmet, vest, gloves, shoes**, lalu menghasilkan `best.pt` yang disalin ke `backend/weights/best.pt` (path aktif `models.ppe` di `backend/config.yaml`).

Jalankan di **Google Colab** dengan GPU: menu *Runtime → Change runtime type → GPU (T4)*. Training di CPU tidak praktis.

**Keamanan:** notebook ini TIDAK menyimpan API key apa pun. Kalau mengunduh dataset dari Roboflow, tempel API key lewat prompt input saat sel dijalankan (tidak tersimpan di file). Jangan menulis key langsung ke sel mana pun sebelum menyimpan/membagikan notebook.

Alur: pasang dependensi → ambil dataset → gabung jadi 4 kelas (`merge_datasets.py`) → latih YOLO11s (`train.py`) → evaluasi mAP per kelas → unduh `best.pt`.

Catatan lisensi: dataset **SH17 berlisensi non-komersial** — hanya untuk edukasi/portfolio, jangan dipakai komersial.

## 1. Cek GPU dan pasang dependensi

In [ ]:
!nvidia-smi
# ultralytics menarik torch yang sudah ada di Colab. roboflow hanya jika dataset dari Roboflow.
!pip -q install ultralytics roboflow pyyaml

## 2. Ambil script repo

Butuh `merge_datasets.py` dan `train.py` dari folder `training/` repo ini. Pilih salah satu cara di sel berikut: clone repo publik, atau unggah manual dua file itu lewat panel Files Colab.

In [ ]:
import os

# Opsi A: clone dari repo (ganti URL bila repo sudah dipublikasikan).
REPO_URL = ""  # mis. "https://github.com/arielshakaramiro/apd-vision.git"; kosongkan jika mau unggah manual
if REPO_URL:
    !git clone --depth 1 {REPO_URL} apd-vision
    os.chdir("apd-vision/training")
else:
    # Opsi B: buat folder kerja; unggah merge_datasets.py dan train.py lewat panel Files ke /content/training
    os.makedirs("/content/training", exist_ok=True)
    os.chdir("/content/training")
    print("Unggah merge_datasets.py dan train.py ke folder ini (panel Files di kiri), lalu lanjut.")
print("Folder kerja:", os.getcwd())
assert os.path.exists("merge_datasets.py") and os.path.exists("train.py"), \
    "merge_datasets.py / train.py belum ada di folder ini"

## 3. Unduh dataset

Butuh minimal satu dataset berisi kelas APD. Dua sumber yang disebut di README repo:

- **SH17** (helmet, vest, gloves, dll.) — unduh dari Releases di `github.com/ahmadmughees/SH17dataset`, lalu susun sebagai folder format YOLO.
- **Roboflow Universe** — banyak dataset "PPE/construction safety". Ekspor format **YOLOv8**.

Tiap sumber harus punya `data.yaml` + folder gambar/label (gaya Roboflow `train/images` atau gaya Ultralytics `images/train`). `merge_datasets.py` menangani keduanya.

Sel berikut contoh unduh dari Roboflow. API key diminta lewat `getpass` saat dijalankan — **tidak** tersimpan di notebook.

In [ ]:
# Contoh Roboflow (lewati sel ini jika memakai SH17 / dataset yang diunggah manual).
from getpass import getpass
from roboflow import Roboflow

api_key = getpass("Roboflow API key (tidak disimpan): ")
rf = Roboflow(api_key=api_key)
# Ganti workspace/project/versi sesuai dataset pilihanmu di Roboflow Universe.
project = rf.workspace("<workspace>").project("<project>")
dataset = project.version(1).download("yolov8", location="/content/datasets/roboflow-ppe")
print("Dataset di:", dataset.location)
del api_key  # jangan biarkan key mengendap di memori sel

In [ ]:
# Jika memakai SH17 atau dataset yang diunggah manual, susun di /content/datasets/<nama>
# dengan data.yaml + gambar/label format YOLO, lalu daftarkan path-nya di sel berikutnya.
!ls -R /content/datasets 2>/dev/null | head -40

## 4. Gabung dataset menjadi 4 kelas

`merge_datasets.py` membaca nama kelas tiap sumber, memetakannya ke `helmet/vest/gloves/shoes`, membuang kelas lain, dan menulis `datasets/ppe4/data.yaml`. Isi `--src` dengan semua folder dataset yang kamu unduh.

In [ ]:
# Sesuaikan daftar sumber dengan dataset yang benar-benar kamu unduh.
SOURCES = "/content/datasets/roboflow-ppe"  # mis. "/content/datasets/sh17 /content/datasets/roboflow-ppe"
!python merge_datasets.py --src {SOURCES} --out /content/datasets/ppe4
!cat /content/datasets/ppe4/data.yaml

## 5. (Opsional) Copy-paste augmentation untuk kelas lemah

Sarung tangan dan sepatu sering kurang data. `copy_paste_augment.py` (unggah juga bila mau memakai) menambah contoh sintetis hanya di split train. Lewati jika belum perlu; buktikan manfaatnya dengan membandingkan mAP per kelas (tanpa vs dengan).

In [ ]:
# !python copy_paste_augment.py --data /content/datasets/ppe4/data.yaml --classes gloves shoes --out /content/datasets/ppe4_cp

## 6. Latih YOLO11s (imgsz 960)

`train.py` melatih dengan `yolo11s.pt` (diunduh otomatis), imgsz 960 (objek kecil seperti sarung tangan sering luput di 640), lalu mengevaluasi split test dan mencetak mAP per kelas. Bobot ada di `runs/ppe4/weights/best.pt`.

Turunkan `--epochs` untuk uji cepat; naikkan untuk hasil lebih baik. `patience=20` menghentikan lebih awal bila tidak membaik.

In [ ]:
!python train.py --data /content/datasets/ppe4/data.yaml --weights yolo11s.pt --imgsz 960 --epochs 100 --batch 16

## 7. Evaluasi ulang per kelas (untuk dicatat di dokumen)

`train.py` sudah mencetak mAP per kelas. Sel ini mengulang evaluasi secara eksplisit supaya angkanya mudah disalin ke `docs/PRD.md` dan `README.md` — catat **apa adanya**, jangan dibulatkan ke atas.

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/ppe4/weights/best.pt")
metrics = model.val(data="/content/datasets/ppe4/data.yaml", split="test", imgsz=960)
per_class = dict(zip(model.names.values(), metrics.box.maps.round(3)))
print("mAP50-95 per kelas:", per_class)
print("mAP50-95 rata-rata:", round(float(metrics.box.map), 3))
print("mAP50 rata-rata   :", round(float(metrics.box.map50), 3))

## 8. Unduh `best.pt` → taruh di repo

Setelah terunduh:
1. Salin file ke `backend/weights/best.pt` di repo lokalmu (atau ubah `models.ppe` di `backend/config.yaml` ke path bobot baru).
2. Buka `backend/config.yaml` → cocokkan `class_map` dengan nama kelas di `data.yaml` hasil merge (urutannya: `helmet, vest, gloves, shoes`).
3. Naikkan `models.version` (mis. `v2` → `v3`) tiap kali melatih ulang, supaya precision per versi bisa dibandingkan.
4. Uji lokal: `python -m scripts.process_video --source data/videos/11.mp4 --start 2026-09-30T08:00`.
5. Catat mAP per kelas (dari sel 7) di `docs/PRD.md` dan `README.md` — hanya angka dari run ini.

`best.pt` tidak di-commit ke git (`*.pt` ada di `.gitignore`).

In [ ]:
from google.colab import files
files.download("runs/ppe4/weights/best.pt")